# Method Choice Examples

**Project question:** How can the same table support different valid analyses when the project questions differ?

By the end of this notebook, you should be able to:

- map continuous, binary, dimension-reduction, and segmentation questions to methods
- pair each method with an evaluation or interpretation standard
- limit claims to the study design and intended use

This notebook is a demonstration, not a homework assignment. The data are
synthetic and were generated for teaching; numerical results should not be
interpreted as evidence about a real organization or population.

In [1]:

from lite_setup import ensure_packages
await ensure_packages()

Using the current Python environment.


In [2]:

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
if not DATA.exists():
    raise FileNotFoundError("Open this notebook from the JupyterLite files root so data/ is available.")

In [3]:
from sklearn.model_selection import cross_val_score, cross_validate, KFold, StratifiedKFold
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [4]:
df = pd.read_csv(DATA / 'project_methods_examples.csv')
df.head()

,id,x1,x2,x3,continuous_outcome,binary_outcome
0,1,0.105,-0.180,1.339,15.695,0
1,2,-0.653,0.351,0.281,16.036,1
2,3,0.829,-0.727,0.867,23.157,1
3,4,1.892,1.649,0.845,23.239,0
4,5,0.294,1.481,0.083,14.415,0


We first define the question and response. Method choice does not begin with whichever algorithm is most advanced.

In [5]:
X = df[['x1', 'x2', 'x3']]
decision_map = pd.DataFrame([
    {
        'question': 'Predict a continuous outcome', 'response': 'continuous_outcome',
        'method': 'linear regression', 'validation': 'RMSE with shuffled K-fold CV',
        'claim': 'out-of-sample predictive association',
    },
    {
        'question': 'Predict a binary outcome', 'response': 'binary_outcome',
        'method': 'logistic regression', 'validation': 'stratified CV; ROC-AUC and balanced accuracy',
        'claim': 'probability ranking/prediction',
    },
    {
        'question': 'Summarize predictor variation', 'response': 'none',
        'method': 'PCA', 'validation': 'explained variance and loading interpretation',
        'claim': 'descriptive low-dimensional structure',
    },
    {
        'question': 'Explore segments', 'response': 'none',
        'method': 'K-means', 'validation': 'silhouette, stability, and profiles',
        'claim': 'exploratory grouping',
    },
])
decision_map

,question,response,method,validation,claim
0,Predict a continuous outcome,continuous_outcome,linear regression,RMSE with shuffled K-fold CV,out-of-sample predictive association
1,Predict a binary outcome,binary_outcome,logistic regression,stratified CV; ROC-AUC and balanced accuracy,probability ranking/prediction
2,Summarize predictor variation,none,PCA,explained variance and loading interpretation,descriptive low-dimensional structure
3,Explore segments,none,K-means,"silhouette, stability, and profiles",exploratory grouping


In [6]:
# Continuous response question: use regression.
y_cont = df['continuous_outcome']
reg_cv = KFold(n_splits=5, shuffle=True, random_state=4031)
reg_fold_rmse = -cross_val_score(
    LinearRegression(), X, y_cont,
    scoring='neg_root_mean_squared_error', cv=reg_cv,
)
pd.Series({
    'cv_rmse_mean': reg_fold_rmse.mean(),
    'cv_rmse_sd': reg_fold_rmse.std(ddof=1),
})

cv_rmse_mean    2.216862
cv_rmse_sd      0.103509
dtype: float64

RMSE is in continuous-outcome units. It estimates predictive error under the chosen fold design; it does not establish that changing a predictor would cause the outcome to change.

In [7]:
# Binary response question: use logistic regression.
y_bin = df['binary_outcome']
logit_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=4031)
logit_scores = cross_validate(
    make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=4031)),
    X, y_bin, scoring={'roc_auc': 'roc_auc', 'balanced_accuracy': 'balanced_accuracy'},
    cv=logit_cv,
)
pd.DataFrame({
    'metric': ['ROC-AUC', 'balanced accuracy'],
    'cv_mean': [logit_scores['test_roc_auc'].mean(), logit_scores['test_balanced_accuracy'].mean()],
    'cv_sd': [logit_scores['test_roc_auc'].std(ddof=1), logit_scores['test_balanced_accuracy'].std(ddof=1)],
})

,metric,cv_mean,cv_sd
0,ROC-AUC,0.733951,0.076054
1,balanced accuracy,0.650000,0.057601


The binary-response method changes because the response and evaluation goal changed, not because the predictor table changed. A final action threshold would still require a stated cost or operational rule.

In [8]:
# Dimension reduction question: use PCA.
pca = PCA(n_components=2)
scores = pca.fit_transform(StandardScaler().fit_transform(X))
pca.explained_variance_ratio_

array([0.51716098, 0.33271567])

PCA has no response here. Explained variance describes compression of the predictor cloud; it is not a prediction score and the components are not causal variables.

In [9]:
# Segmentation question: use K-means and profile clusters.
X_scaled = StandardScaler().fit_transform(X)
km = KMeans(n_clusters=3, random_state=4031, n_init=20)
df['cluster'] = km.fit_predict(X_scaled)
print(f'Silhouette: {silhouette_score(X_scaled, df["cluster"]):.3f}')
df.groupby('cluster')[['x1', 'x2', 'x3', 'continuous_outcome', 'binary_outcome']].mean().round(2)

Silhouette: 0.293


,x1,x2,x3,continuous_outcome,binary_outcome
cluster,,,,,
0,0.94,1.09,0.05,21.43,0.58
1,-0.66,-0.47,0.76,18.16,0.29
2,-0.17,-0.46,-1.09,20.25,0.69


Outcomes are shown only for post-hoc cluster profiling; they were not used to create the clusters. Cluster IDs are arbitrary, and a useful segmentation claim requires stability and domain relevance beyond one silhouette value.

**Transfer exercise:** Write your project question in one sentence. Identify the response type, whether rows are independent or time/group ordered, the intended use (prediction, explanation, description, or causal inference), one primary metric, and one claim your design cannot support.